In [ ]:
#On omega with NoMachine
#$ ssh -XY somega.gat.com
#$ module load fuse
#$ fuse  #if you want to run from the terminal
#$ jupyter-lab # if you want to run from a notebook (preferred)

using Revise
using Plots              # for plotting
using FUSE               # this will also import IMAS in the current namespace
using TurbulentTransport
using TJLF
# Pkg.status()

Run TJLF at one radial location using the existing `dd` / `act` and reconstruct
the 2D potential fluctuation spectrum `φ(kx, ky, mode)` with
`TurbulentTransport.fluctuation_spectra`. The reconstruction uses the Staebler
spectral-shift Lorentzian kx distribution, so it requires `SAT_RULE ∈ {1, 2, 3}`
and `ALPHA_QUENCH = 0`.

In [ ]:
# --- 0) Make sure we have a dd to work with (re-use an earlier one if available) ---
if !@isdefined(dd)
    ini, act = FUSE.case_parameters(:D3D, :L_mode);
    global dd = IMAS.dd();
    FUSE.init(dd, ini, act);
end
# --- 1) Build the InputTJLF at a chosen ρ ---
# Fast path: cell 29 already caches `input_tjlf` in TORCUT_SINGLE_CACHE.
# If that file exists we load it directly (near-instant on restart).
# Delete the cache file or set force_rebuild_input = true to regenerate.
#
# On a cold start (no cache) we build InputTGLF → InputTJLF from dd WITHOUT
# running any TJLF eigenvalue solve: cell 29 owns that work and caches it.
using Serialization
import TJLF
force_rebuild_input = false
TORCUT_SINGLE_CACHE_28 = joinpath("tjlf_cache", "torcut_single.bin")

rho_plot = 0.6
act.ActorTGLF.model           = :TJLF
act.ActorTGLF.sat_rule        = :sat2
act.ActorTGLF.electromagnetic = true
act.ActorTGLF.lump_ions       = true

if !force_rebuild_input && isfile(TORCUT_SINGLE_CACHE_28)
    @info "Reusing InputTJLF from cache (instant)" TORCUT_SINGLE_CACHE_28
    _c28       = deserialize(TORCUT_SINGLE_CACHE_28)
    input_tjlf = _c28.input_tjlf
    rho_plot   = _c28.rho_plot
else
    @info "No cache — building InputTJLF from dd (no TJLF solve; cell 29 will run & cache it)"
    _inp_tglf = TurbulentTransport.InputTGLF(dd, [Float64(rho_plot)], :sat2, true, true; MXH_modes=1)[1]
    input_tjlf = TJLF.InputTJLF{Float64}(_inp_tglf)
    input_tjlf.ALPHA_QUENCH = 0.0
end


In [ ]:
# --- 2) Run TJLF & reconstruct |φ|²(kx, ky, mode) ---
# Auto-cache: skip the heavy `fluctuation_spectra` + `TJLF.run` when
# `tjlf_cache/torcut_single.bin` exists. Set `force_recompute_single = true`
# (or delete the file) to recompute. The cache stores everything cells 31 + 35
# need, including the eigenvalue array (γ, ω) used by the movie.
using Serialization
import TJLF
import GACODE
TORCUT_SINGLE_CACHE = joinpath("tjlf_cache", "torcut_single.bin")
force_recompute_single = false

if !force_recompute_single && isfile(TORCUT_SINGLE_CACHE)
    @info "Loading single-ρ TJLF cache" TORCUT_SINGLE_CACHE
    cache       = deserialize(TORCUT_SINGLE_CACHE)
    input_tjlf  = cache.input_tjlf
    fs          = cache.fs
    tjlf_eigval = cache.eigenvalue
    csa_rho     = cache.csa_rho
    csa_prof    = cache.csa_prof
    rho_plot    = cache.rho_plot
    a_m         = cache.a_m
else
    tjlf_result = TJLF.run(input_tjlf)
    fs          = TurbulentTransport.fluctuation_spectra(tjlf_result, input_tjlf;
                                                         n_kx=201, kx_max_sigma=4.0)
    tjlf_eigval = tjlf_result.eigenvalue
    _cp1d_29    = dd.core_profiles.profiles_1d[]
    _a_m_29     = dd.equilibrium.time_slice[].boundary.minor_radius
    csa_rho     = collect(Float64, _cp1d_29.grid.rho_tor_norm)
    csa_prof    = GACODE.c_s(_cp1d_29) .* 1e-2 ./ _a_m_29
    mkpath(dirname(TORCUT_SINGLE_CACHE))
    serialize(TORCUT_SINGLE_CACHE,
        (; input_tjlf, fs, eigenvalue = tjlf_eigval,
           csa_rho, csa_prof, rho_plot, a_m = _a_m_29))
    @info "Saved single-ρ TJLF cache" TORCUT_SINGLE_CACHE
end

phi2_tot = dropdims(sum(fs.phi2;     dims=3); dims=3)   # sum over modes → (nkx, nky)
phi2_ky  = vec(sum(fs.phi2_ky;       dims=2))           # phinorm summed over modes → (nky,)
@info "TJLF fluctuation spectrum" rho=rho_plot nkx=length(fs.kx) nky=length(fs.ky) nmodes=size(fs.phi2, 3) SAT_RULE=fs.sat_rule

# --- 3) Plot |φ|²(kx, ky) (linear + log) and the ky marginal ---
p_lin = heatmap(fs.ky, fs.kx, phi2_tot;
    xlabel="k_θ ρ_s", ylabel="k_x ρ_s",
    title="|φ|²(k_x, k_y)   ρ = $(rho_plot),  SAT$(Int(input_tjlf.SAT_RULE))",
    colorbar_title="|φ|²",
    xscale=:log10)
plot!(p_lin, fs.ky, fs.kx0_e; lw=2, color=:white, label="k_x0,e(k_y)")

p_log = heatmap(fs.ky, fs.kx, log10.(max.(phi2_tot, 1e-20));
    xlabel="k_θ ρ_s", ylabel="k_x ρ_s",
    title="log₁₀ |φ|²(k_x, k_y)",
    colorbar_title="log₁₀ |φ|²",
    xscale=:log10)
plot!(p_log, fs.ky, fs.kx0_e; lw=2, color=:white, label="k_x0,e(k_y)")

p_marg = plot(fs.ky, phi2_ky;
    xlabel="k_θ ρ_s", ylabel="|φ|²(k_x = 0, k_y)  (phinorm)",
    title="k_y marginal",
    xscale=:log10, yscale=:log10, legend=false, lw=2)

plot(p_lin, p_log, p_marg; layout=(1, 3), size=(1500, 420))



Build a synthetic real-space realization of the potential fluctuations and
render it on the tokamak poloidal cross-section (like CGYRO's
`cgyro_plot -vis torcut`, but from a quasi-linear TJLF run).

The procedure is:

1. Get the exact `ρ_s/a` at `ρ_plot` from the dd using
   `GACODE.rho_s(cp1d, eqt)` and `eqt.boundary.minor_radius`.
   TJLF's spectral grid is dimensionless (`kx ρ_s`, `ky ρ_s`), so this is the
   physical scale that converts wavenumbers to real-space eddy sizes. No
   free visual tuning of `ρ_s` is needed.
2. Draw independent random phases `ψ(kx, ky)` for every Fourier component and
   weight them by `√|φ|²(kx, ky)` from the previous cell.
3. Multiply by a Gaussian ballooning envelope `exp(-θ²/(2 θ_w²))` so the
   reconstruction is peaked on the low-field side, where unstable drift modes
   live.
4. Apply the standard field-line-following binormal coordinate with magnetic
   shear, so the eddies "tilt" as a function of poloidal angle:
   `y(x, θ) / ρ_s = -q·[r₀/ρ_s + (1+ŝ)·x/ρ_s]·θ`.
5. Map the local `(r, θ)` field to `(R, Z)` using the Miller geometry carried
   by `input_tjlf` and rasterize onto a rectangular pixel grid. Pixels outside
   the annulus are set to `NaN` (rendered as blank) so no spurious
   interpolation crosses the "hole" in the flux tube — the same masking trick
   that `cgyro_plot` uses with matplotlib's `tripcolor`.


In [ ]:
# ============================================================================
# "torcut" projection of |φ|² onto the poloidal cross-section
# ============================================================================
using Interpolations, Random
using Plots.PlotMeasures
using LaTeXStrings
using Printf
import GACODE

# --- user-tunable knobs ---
Random.seed!(123)                 # deterministic realization of random phases
radial_half_width  = 0.35         # ± around ρ_plot to visualize, in units of a
ky_cutoff          = 2.0          # discard ky·ρ_s > this (ETG tail) for clarity
nr_mesh            = 120          # radial (r, θ) samples for reconstruction
ntheta_mesh        = 320          # poloidal samples
nR_pix             = 400          # horizontal raster pixels
nZ_pix             = 560          # vertical raster pixels

# --- exact ρ_s/a at ρ_plot from the dd (TGLF convention: ρ_s = c_s / Ω_ci,
#     built with B_unit = (q/r) dψ/dr and a = eqt.boundary.minor_radius) ---
eqt1d        = dd.equilibrium.time_slice[].profiles_1d
cp1d         = dd.core_profiles.profiles_1d[]
eqt          = dd.equilibrium.time_slice[]
rho_s_cm     = GACODE.rho_s(cp1d, eqt)                         # [cm] profile on cp1d.grid
a_m          = eqt.boundary.minor_radius                       # [m]
rho_s_over_a = IMAS.interp1d(cp1d.grid.rho_tor_norm, rho_s_cm .* 1e-2 ./ a_m)(rho_plot)
@info "Physical normalization at ρ=$(rho_plot)" ρs_over_a=rho_s_over_a ρs_cm=IMAS.interp1d(cp1d.grid.rho_tor_norm, rho_s_cm)(rho_plot) a_m

# --- 1) Local Miller geometry from the InputTJLF at ρ_plot ---
# Includes Shafranov shift (DRMAJDX_LOC, DZMAJDX_LOC) and shape gradients
# (S_KAPPA_LOC, S_DELTA_LOC, S_ZETA_LOC) so flux surfaces vary with r.
rmaj   = input_tjlf.RMAJ_LOC
zmaj   = input_tjlf.ZMAJ_LOC
kappa  = input_tjlf.KAPPA_LOC
delta  = input_tjlf.DELTA_LOC
zeta   = input_tjlf.ZETA_LOC
rmin0  = input_tjlf.RMIN_LOC
q0     = abs(input_tjlf.Q_LOC)
shat   = input_tjlf.Q_PRIME_LOC * rmin0^2 / q0^2     # magnetic shear s = r/q · dq/dr
dRdr   = input_tjlf.DRMAJDX_LOC                      # ∂R₀/∂r (Shafranov shift)
dZdr   = input_tjlf.DZMAJDX_LOC                      # ∂Z₀/∂r
sκ     = input_tjlf.S_KAPPA_LOC                      # (r/κ)·∂κ/∂r
sδ     = input_tjlf.S_DELTA_LOC                      # r·∂δ/∂r (TJLF argR_r convention)
sζ     = input_tjlf.S_ZETA_LOC                       # r·∂ζ/∂r
asd0   = asin(delta)
cd0    = cos(asd0)                                   # √(1−δ²)
θw_ky  = collect(Float64, input_tjlf.WIDTH_SPECTRUM) # per-ky ballooning envelope width [rad]

# Forward Miller map with linear-in-r extrapolation of (R₀, Z₀, κ, δ, ζ)
# from the local TJLF gradients. MXH harmonics beyond (δ, ζ) are ignored.
function miller_RZ(r, θ)
    Δr  = r - rmin0
    R0r = rmaj  + dRdr * Δr
    Z0r = zmaj  + dZdr * Δr
    κr  = kappa * (1 + sκ * Δr / rmin0)
    asd = asd0  + sδ * Δr / (rmin0 * cd0)            # arcsin(δ(r))
    ζr  = zeta  + sζ * Δr / rmin0
    return (R0r + r * cos(θ + asd*sin(θ) - ζr*sin(2θ)),
            Z0r + κr * r * sin(θ))
end

# 2-D damped Newton inverse: solves F(r, θ) = (R(r,θ)−Rq, Z(r,θ)−Zq) = 0.
# Initial guess: concentric Miller (Δr = 0) inversion, then refine with
# the full radially-varying Jacobian.
function invert_miller(Rq, Zq; iters=20, tol=1e-10)
    s_r = Rq - rmaj
    s_z = (Zq - zmaj) / kappa
    θ   = atan(s_z, s_r)
    asd_g = asd0
    for _ in 1:6
        a  = θ + asd_g*sin(θ) - zeta*sin(2θ)
        ap = 1 + asd_g*cos(θ) - 2*zeta*cos(2θ)
        f  = s_z*cos(a) - s_r*sin(θ)
        df = -s_z*sin(a)*ap - s_r*cos(θ)
        abs(df) < 1e-14 && break
        Δ = f / df
        abs(Δ) > 0.5 && (Δ = 0.5 * sign(Δ))
        θ -= Δ
        abs(Δ) < 1e-8 && break
    end
    a0 = θ + asd_g*sin(θ) - zeta*sin(2θ)
    r  = (abs(cos(a0)) >= abs(sin(θ))) ? s_r/cos(a0) : s_z/sin(θ)
    r < 0 && (r = -r; θ += π)

    for _ in 1:iters
        Δr   = r - rmin0
        κr   = kappa * (1 + sκ * Δr / rmin0)
        dκdr = kappa * sκ / rmin0
        asd  = asd0 + sδ * Δr / (rmin0 * cd0)
        ζr   = zeta + sζ * Δr / rmin0
        dasd = sδ / (rmin0 * cd0)
        dζdr = sζ / rmin0
        argR    = θ + asd*sin(θ) - ζr*sin(2θ)
        dargRdr = dasd*sin(θ) - dζdr*sin(2θ)
        dargRdθ = 1 + asd*cos(θ) - 2*ζr*cos(2θ)
        cR, sR = cos(argR), sin(argR)
        F1 = (rmaj + dRdr*Δr) + r*cR - Rq
        F2 = (zmaj + dZdr*Δr) + κr*r*sin(θ) - Zq
        J11 = dRdr + cR - r*sR*dargRdr
        J12 = -r*sR*dargRdθ
        J21 = dZdr + (κr + dκdr*r)*sin(θ)
        J22 = κr*r*cos(θ)
        det = J11*J22 - J12*J21
        abs(det) < 1e-14 && break
        δr = (J22*F1 - J12*F2) / det
        δθ = (J11*F2 - J21*F1) / det
        s = max(abs(δr) / 0.1, abs(δθ) / 0.5, 1.0)   # damp big steps
        r -= δr / s
        θ -= δθ / s
        max(abs(δr), abs(δθ)) < tol && break
    end
    if r < 0
        r = -r
        θ += π
    end
    return r, mod(θ + π, 2π) - π
end

# --- 2) Reconstruct φ(r, θ) from |φ|²(kx, ky) ---
r_in, r_out = max(rmin0 - radial_half_width, 1e-3), min(rmin0 + radial_half_width, 0.99)
r_grid      = collect(range(r_in,  r_out; length=nr_mesh))
theta_grid  = collect(range(-π,   π;     length=ntheta_mesh))
x_rs        = (r_grid .- rmin0) ./ rho_s_over_a        # local radial coord, in ρ_s
r0_rs       = rmin0 / rho_s_over_a
env_θky    = [exp(-θ^2 / (2 * θw_ky[j]^2)) for θ in theta_grid, j in eachindex(θw_ky)]

phi_amp = sqrt.(max.(dropdims(sum(fs.phi2; dims=3); dims=3), 0))  # (nkx, nky), modes summed
ky_use  = findall(j -> fs.ky[j] ≤ ky_cutoff, eachindex(fs.ky))
phases  = 2π .* rand(length(fs.kx), length(fs.ky))

@info "torcut reconstruction" ρ=rho_plot q0 shat rmin0 θw_ky_min=minimum(θw_ky) θw_ky_max=maximum(θw_ky) θw_ky_mean=sum(θw_ky)/length(θw_ky) nky_used=length(ky_use) ky_max_used=fs.ky[ky_use[end]]

# Binormal phase at fixed toroidal angle φ=0: for TGLF's poloidal wavenumber
# ky·ρ_s = n·q·ρ_s/r, a single toroidal-n drift-wave has phase -n·q(r)·θ at (r, θ).
# Substituting n·q = ky·(r/ρ_s) and q(r) ≈ q₀·(1+ŝ·x/r₀), the q factor cancels:
#   phase(kx, ky; x, θ) = (kx - ky·ŝ·θ)·(x/ρ_s)  -  ky·(r₀/ρ_s)·θ  +  ψ
# Note: the lowest ky·ρ_s in the TGLF spectrum tiles the pattern
# ky·(r₀/ρ_s) times around the poloidal direction — this *is* the flux-tube
# binormal periodicity mapped into the torus.
# Typed, threaded reconstruction of φ(r, θ) — hoisted out of Jupyter's
# global scope to avoid per-iteration dynamic dispatch. θ-rows are independent,
# so `Threads.@threads` is safe with no data race on `phi_rt`.
function _reconstruct_phi_rt!(phi_rt::Matrix{Float64}, theta_grid::Vector{Float64},
        env_θky::Matrix{Float64}, kx::Vector{Float64}, ky::Vector{Float64},
        phi_amp::Matrix{Float64}, phases::Matrix{Float64},
        x_rs::Vector{Float64}, r0_rs::Float64, shat::Float64,
        ky_use::Vector{Int}, nr_mesh::Int, ntheta_mesh::Int)
    Threads.@threads for k in 1:ntheta_mesh
        θ = theta_grid[k]
        phase_θ  = -r0_rs * θ
        kx_shift = -shat  * θ
        @inbounds for j in ky_use
            env = env_θky[k, j]
            env == 0 && continue
            kyj = ky[j]
            pθ  = kyj * phase_θ
            Δkx = kyj * kx_shift
            for ikx in eachindex(kx)
                amp = phi_amp[ikx, j]
                amp == 0 && continue
                ktot = kx[ikx] + Δkx
                ψ    = phases[ikx, j]
                for i in 1:nr_mesh
                    phi_rt[i, k] += amp * cos(ktot * x_rs[i] + pθ + ψ) * env
                end
            end
        end
    end
    return phi_rt
end
phi_rt = zeros(Float64, nr_mesh, ntheta_mesh)
_reconstruct_phi_rt!(phi_rt, collect(Float64, theta_grid), Matrix{Float64}(env_θky),
    collect(Float64, fs.kx), collect(Float64, fs.ky),
    Matrix{Float64}(phi_amp), Matrix{Float64}(phases),
    collect(Float64, x_rs), Float64(r0_rs), Float64(shat),
    collect(Int, ky_use), Int(nr_mesh), Int(ntheta_mesh))
phi_rt ./= maximum(abs, phi_rt)

# --- 3) Rasterize φ(r, θ) on a rectangular (R, Z) grid, masking outside the band ---
itp_phi = Interpolations.extrapolate(
    Interpolations.interpolate((r_grid, theta_grid), phi_rt, Gridded(Linear())),
    (Flat(), Periodic()))

Rmin_bb = minimum(miller_RZ(r_out, θ)[1] for θ in theta_grid) - 0.03
Rmax_bb = maximum(miller_RZ(r_out, θ)[1] for θ in theta_grid) + 0.03
Zmin_bb = minimum(miller_RZ(r_out, θ)[2] for θ in theta_grid) - 0.03
Zmax_bb = maximum(miller_RZ(r_out, θ)[2] for θ in theta_grid) + 0.03
Rpix = collect(range(Rmin_bb, Rmax_bb; length=nR_pix))
Zpix = collect(range(Zmin_bb, Zmax_bb; length=nZ_pix))
function _raster_torcut!(phi_raster::Matrix{Float64}, Rpix::Vector{Float64},
        Zpix::Vector{Float64}, itp_phi, invert_miller, r_in::Float64, r_out::Float64)
    nR, nZ = length(Rpix), length(Zpix)
    Threads.@threads for jp in 1:nZ
        @inbounds for ip in 1:nR
            r, θ = invert_miller(Rpix[ip], Zpix[jp])
            if r_in ≤ r ≤ r_out
                phi_raster[ip, jp] = itp_phi(r, θ)
            end
        end
    end
    return phi_raster
end
phi_raster = fill(NaN, nR_pix, nZ_pix)
_raster_torcut!(phi_raster, collect(Float64, Rpix), collect(Float64, Zpix),
    itp_phi, invert_miller, Float64(r_in), Float64(r_out))

# --- 4) Plot on (R, Z) ---
vmax = maximum(abs, filter(!isnan, phi_raster))
ρ_str = @sprintf("%.2f", rho_plot)
sat_n = Int(input_tjlf.SAT_RULE)
p_torcut = heatmap(Rpix, Zpix, phi_raster';
    c = cgrad(:RdBu_10, rev=true), clims = (-vmax, vmax),
    aspect_ratio = :equal, colorbar = false, legend = false, framestyle = :box,
    xlabel = L"R / a", ylabel = L"Z / a", size = (540, 760),
    left_margin = 4mm, right_margin = 4mm, bottom_margin = 4mm, top_margin = 4mm,
    fontfamily = "Computer Modern",
    titlefontfamily = "Computer Modern",
    guidefontfamily = "Computer Modern",
    tickfontfamily  = "Computer Modern",
    title = L"|\varphi|\ \ (\rho=%$ρ_str,\ \mathrm{SAT}%$sat_n)")

# bounding flux surfaces
R_in_line  = [miller_RZ(r_in,  θ)[1] for θ in theta_grid]
Z_in_line  = [miller_RZ(r_in,  θ)[2] for θ in theta_grid]
R_out_line = [miller_RZ(r_out, θ)[1] for θ in theta_grid]
Z_out_line = [miller_RZ(r_out, θ)[2] for θ in theta_grid]
R_rho_line = [miller_RZ(rmin0, θ)[1] for θ in theta_grid]
Z_rho_line = [miller_RZ(rmin0, θ)[2] for θ in theta_grid]
plot!(p_torcut, R_in_line,  Z_in_line;  color=:black, lw=1.4)
plot!(p_torcut, R_out_line, Z_out_line; color=:black, lw=1.4)
plot!(p_torcut, R_rho_line, Z_rho_line; color=:black, lw=0.8, ls=:dash)
p_torcut


Extend the single-ρ torcut to a stack of TJLF runs at `rho_list`. Each ρᵢ owns a
radial band `[r_{i−½}, r_{i+½}]` (midpoints between neighbouring `RMIN_LOC`s)
and contributes its **own** Miller geometry, SAT spectrum, `ρ_s`, magnetic
shear and ballooning width. Inside its band, the synthetic field is
reconstructed exactly as in the single-ρ cell (same phase formula
`(kx − ky·ŝ·θ)·(x/ρ_s) − ky·(r₀/ρ_s)·θ`), with an *independent* random-phase
realization. The bands are then stitched into a single `(R, Z)` raster via
the middle band's Miller map as a global coordinate, which introduces only a
small geometric error because neighbouring local Miller shapes differ
slightly.

Notes / possible extensions:

- Each band is normalised to unit peak so the visualisation does not
  disappear in low-flux regions. For a physically-weighted picture, scale by
  the local gyro-Bohm flux level
  (`GACODE.gyrobohm_energy_flux(ne, Te, ρ_s, a)`) instead.
- Adjacent bands are blended with a cosine weight `w_lo = cos(π t/2)`,
  `w_hi = sin(π t/2)` between their centres (`t ∈ [0, 1]`). Because the two
  realisations use *independent* random phases, `w_lo² + w_hi² = 1`
  preserves the variance of `φ` uniformly across the seam (as opposed to
  linear `w_lo + w_hi = 1`, which would dip by `√2/2` at the mid-point).
  Each band's `(r, θ)` interpolator is built on a grid that reaches to the
  two neighbouring band centres so the blend always uses valid data.
- Using the dd equilibrium's ψ(R, Z) (via `IMAS` flux-surface interpolation)
  instead of a global Miller map would give exact flux-surface coordinates
  for non-Miller-like equilibria.

The ρ-grid is sized from the **radial correlation length** of the turbulence:
`TurbulentTransport.radial_correlation_length(fs)` returns the HWHM of
`|C(Δr; ky)|`, the inverse-Fourier transform of the kx power spectrum at fixed
ky. The intensity-weighted average `⟨L_r⟩/ρ_s` sets the physical decorrelation
scale; mapped to units of the minor radius via `⟨L_r⟩/a = (⟨L_r⟩/ρ_s)·(ρ_s/a)`,
we ask `Δρ ≈ ⟨L_r⟩/a`:

- `Δρ ≳ ⟨L_r⟩/a` — adjacent bands are **statistically independent**, so
  drawing independent random phases per band (the only choice available from a
  quasilinear model) is physically justified. The cosine blend only smooths
  the visual seam; it does not invent phase coherence.
- `Δρ ≪ ⟨L_r⟩/a` — adjacent bands *should* be correlated; our
  independent-phase reconstruction would under-represent that correlation. A
  warning is printed when this happens.

The total number of radii is capped at `n_rho_cap` (default `8`, matching the
user's production thread count). When the physics asks for more than the cap,
a uniform grid of `n_rho_cap` points spans `rho_bounds`; this still keeps
`Δρ ≳ ⟨L_r⟩` in all typical core-plasma cases.


In [ ]:
# ============================================================================
# Multi-ρ torcut: run TJLF at a list of radii and stitch the realisations
# ============================================================================
using Interpolations, Random
import GACODE

# --- user knobs ---
rho_bounds       = (0.2, 0.95)        # range of ρ_tor_norm to cover
n_rho_cap        = 10                # hard cap on number of TJLF evaluations (≤ JULIA_NUM_THREADS)
rho_probe        = 0.5               # ρ at which to probe the kx spectrum to size the grid
ky_cutoff_multi  = 2.0   # hard cap on ky·ρ_s; per-band raster Nyquist is applied on top
ntheta_band_min  = 480   # floor; actual ntheta_band is set adaptively from ky_cutoff·r₀/ρ_s
nr_band          = 192
nR_pix_multi     = 400
nZ_pix_multi     = 560
Random.seed!(456)

# Auto-cache: skip the probe + per-band `fluctuation_spectra` loop when
# `tjlf_cache/torcut_multi.bin` exists. Set `force_recompute_multi = true` (or
# delete the file) to recompute. The cache stores `bands` (with WIDTH_SPECTRUM),
# the per-band eigenvalue arrays (γ, ω) used by the movie cell, and the c_s/a
# profile so cells 34 + 36 work after a fresh kernel restart.
using Serialization
import TJLF
TORCUT_MULTI_CACHE = joinpath("tjlf_cache", "torcut_multi.bin")
force_recompute_multi = true

if !force_recompute_multi && isfile(TORCUT_MULTI_CACHE)
    @info "Loading multi-band TJLF cache" TORCUT_MULTI_CACHE
    cache_m         = deserialize(TORCUT_MULTI_CACHE)
    bands           = cache_m.bands
    band_eigenvalue = cache_m.band_eigenvalue
    rho_list_multi  = cache_m.rho_list_multi
    rmin_centers    = cache_m.rmin_centers
    rmin_edges      = cache_m.rmin_edges
    csa_rho         = cache_m.csa_rho
    csa_prof        = cache_m.csa_prof
    a_m             = cache_m.a_m
else
    # ρ_s/a profile from the dd (used for both grid sizing and per-band reconstruction)
    eqt_m   = dd.equilibrium.time_slice[]
    cp1d_m  = dd.core_profiles.profiles_1d[]
    a_m_m   = eqt_m.boundary.minor_radius
    rs_itp  = IMAS.interp1d(cp1d_m.grid.rho_tor_norm, GACODE.rho_s(cp1d_m, eqt_m) .* 1e-2 ./ a_m_m)

    # --- Size the ρ-grid from the TJLF kx spectrum ---------------------------------
    # The longest radial wavelength carried by `fluctuation_spectra` is
    #       λ_r / a  =  2π · (ρ_s/a) / (kx_min · ρ_s)
    # where kx_min·ρ_s is the smallest nonzero point on the kx grid.  Adjacent
    # bands separated by more than λ_r/a would under-resolve the radial envelope
    # of the longest-wavelength mode, so we take Δρ ≤ λ_r/a and then cap at
    # n_rho_cap for speed.  A single TJLF probe at ρ = rho_probe is enough because
    # kx·ρ_s is weakly ρ-dependent for a given spectral grid.
    act.ActorTGLF.model           = :TJLF
    act.ActorTGLF.sat_rule        = :sat2
    act.ActorTGLF.electromagnetic = true
    act.ActorTGLF.lump_ions       = true

    actor_probe = FUSE.ActorTGLF(dd, act; rho_transport=[rho_probe])
    inp_probe   = deepcopy(actor_probe.input_tglfs[1])
    inp_probe.ALPHA_QUENCH = 0.0
    fs_probe    = TurbulentTransport.fluctuation_spectra(inp_probe; n_kx=201, kx_max_sigma=2.5)

    # --- physical target: 1 radial correlation length per band ---------------------
    # `radial_correlation_length` returns the HWHM of the |C(Δr; ky)| envelope from
    # the inverse FT of the kx power spectrum (ρ_s units). Independent random
    # phases between bands are physically justified only when Δρ ≥ ⟨L_r⟩/a, since
    # the turbulence loses memory over that scale.
    rc        = TurbulentTransport.radial_correlation_length(fs_probe; method=:hwhm)
    ρs_over_a = rs_itp(rho_probe)
    Lr_over_a = rc.L_avg * ρs_over_a                             # ⟨L_r⟩ / a

    # grid sizing
    n_needed  = ceil(Int, (rho_bounds[2] - rho_bounds[1]) / Lr_over_a) + 1
    n_rho     = clamp(n_needed, 3, n_rho_cap)
    rho_list_multi = collect(range(rho_bounds[1], rho_bounds[2]; length=n_rho))
    Δρ_used   = n_rho > 1 ? (rho_bounds[2] - rho_bounds[1]) / (n_rho - 1) : NaN

    # diagnostic: how many correlation lengths between adjacent bands?
    decorr_ratio = Δρ_used / Lr_over_a

    println("─"^60)
    println("TJLF multi-ρ grid sizing  (Julia threads = $(Threads.nthreads()))")
    println("  probe ρ                            = $rho_probe")
    println("  ρ_s/a                              = $(round(ρs_over_a; sigdigits=4))")
    println("  ⟨L_r⟩/ρ_s   (HWHM of |C(Δr)|)      = $(round(rc.L_avg;   sigdigits=4))")
    println("  ⟨L_r⟩/a                            = $(round(Lr_over_a;  sigdigits=4))")
    println("  optimum # of ρ points (Δρ = ⟨L_r⟩) = $n_needed")
    println("  cap (n_rho_cap)                    = $n_rho_cap")
    println("  ⇒ using $n_rho points at Δρ ≈ $(round(Δρ_used; digits=3))  ($(round(decorr_ratio; digits=2))·⟨L_r⟩)")
    println("     rho_list_multi = $rho_list_multi")
    if decorr_ratio < 1
        @warn "Δρ < ⟨L_r⟩ — adjacent bands are correlated; independent-phase blend will under-represent that correlation." Δρ_used Lr_over_a
    end
    println("─"^60)

    # --- Now run the TGLF actor at every ρ in the final grid ---
    actor_multi = FUSE.ActorTGLF(dd, act; rho_transport=rho_list_multi)

    # --- Build per-band (r, θ) reconstruction + interpolator (threaded) ---
    # 1) gather the per-band TJLF inputs & spectra (serial; TJLF already threaded).
    bands           = Vector{NamedTuple}(undef, length(rho_list_multi))
    band_eigenvalue = Vector{Array{Float64,3}}(undef, length(rho_list_multi))
    for (i, ρi) in enumerate(rho_list_multi)
        inp = deepcopy(actor_multi.input_tglfs[i])
        inp.ALPHA_QUENCH = 0.0
        res_i = TJLF.run(inp)
        fs_i  = TurbulentTransport.fluctuation_spectra(res_i, inp; n_kx=201, kx_max_sigma=2.5)
        bands[i]           = (ρ = ρi, input = inp, fs = fs_i, rho_s_over_a = rs_itp(ρi))
        band_eigenvalue[i] = res_i.eigenvalue
    end
    perm = sortperm([b.input.RMIN_LOC for b in bands])
    bands           = bands[perm]
    band_eigenvalue = band_eigenvalue[perm]
    rmin_centers = [b.input.RMIN_LOC for b in bands]
    # Symmetric band edges: mirror the interior midpoint spacing at both ends so
    # the innermost band does not get extrapolated to the magnetic axis (which
    # otherwise produces concentric-ring aliasing in the deep core).
    Δin  = length(rmin_centers) >= 2 ? (rmin_centers[2]   - rmin_centers[1])     / 2 : 0.05
    Δout = length(rmin_centers) >= 2 ? (rmin_centers[end] - rmin_centers[end-1]) / 2 : 0.05
    rmin_edges   = vcat(max(rmin_centers[1]   - Δin,  1e-3),
        [(rmin_centers[i] + rmin_centers[i+1]) / 2 for i in 1:length(bands)-1],
        min(rmin_centers[end] + Δout, 0.999))

    # cache the c_s/a profile so the movie cell (36) doesn't need cp1d after restart
    csa_rho  = collect(Float64, cp1d_m.grid.rho_tor_norm)
    csa_prof = GACODE.c_s(cp1d_m) .* 1e-2 ./ a_m_m
    a_m      = a_m_m

    mkpath(dirname(TORCUT_MULTI_CACHE))
    serialize(TORCUT_MULTI_CACHE,
        (; bands, band_eigenvalue, rho_list_multi, rmin_centers, rmin_edges,
           csa_rho, csa_prof, a_m))
    @info "Saved multi-band TJLF cache" TORCUT_MULTI_CACHE
end

# θ-Nyquist: poloidal samples must resolve the shortest binormal wavelength.
# A mode with wavenumber ky has θ-phase ≈ -ky·(r/ρ_s)·θ, so its number of
# oscillations around the torus is 2π·ky·r/ρ_s. We take ≥4× Nyquist at the
# worst-case band (largest ky·r/ρ_s) and floor at ntheta_band_min.
ntheta_band = let
    worst = maximum(ky_cutoff_multi * b.input.RMIN_LOC / b.rho_s_over_a for b in bands)
    max(ntheta_band_min, 4 * ceil(Int, 2π * worst))
end
@info "Adaptive θ grid" ntheta_band ntheta_band_min worst_ky_r_over_rhos=maximum(ky_cutoff_multi * b.input.RMIN_LOC / b.rho_s_over_a for b in bands)

# 2) pre-draw random phases serially so results are reproducible under threads.
band_phases = [2π .* rand(length(b.fs.kx), length(b.fs.ky)) for b in bands]

θ_g       = collect(range(-π, π; length=ntheta_band))
# bounding-box estimate used by per-band ky-cutoff (same formula as the raster bbox)
Rmin_bb_est = minimum(bands[end].input.RMAJ_LOC + bands[end].input.RMIN_LOC * cos(θ) for θ in θ_g) - 0.03
Rmax_bb_est = maximum(bands[end].input.RMAJ_LOC + bands[end].input.RMIN_LOC * cos(θ) for θ in θ_g) + 0.03
Zmin_bb_est = minimum(bands[end].input.ZMAJ_LOC + bands[end].input.KAPPA_LOC * bands[end].input.RMIN_LOC * sin(θ) for θ in θ_g) - 0.03
Zmax_bb_est = maximum(bands[end].input.ZMAJ_LOC + bands[end].input.KAPPA_LOC * bands[end].input.RMIN_LOC * sin(θ) for θ in θ_g) + 0.03
band_itps = Vector{Any}(undef, length(bands))

# 3) Typed helper: build one band's (r, θ) interpolator.  Moving the hot
#    loop out of Jupyter's global scope avoids dynamic dispatch on every
#    variable access and makes the `Threads.@threads` driver below trivial.
function _build_band_itp(rmin0::Float64, q0::Float64, shat::Float64,
        θw_ky::Vector{Float64},
        rs_a::Float64, r_lo::Float64, r_hi::Float64,
        fs_kx::Vector{Float64}, fs_ky::Vector{Float64},
        phi2::Array{Float64,3}, phases::Matrix{Float64},
        θ_g::Vector{Float64}, nr::Int, k_nyq_rs::Float64, ky_cap::Float64)
    nkx = length(fs_kx); nky = length(fs_ky); ntheta = length(θ_g)
    r_g   = collect(range(r_lo, r_hi; length=nr))
    x_rs  = (r_g .- rmin0) ./ rs_a
    r0_rs = rmin0 / rs_a
    # Per-ky Gaussian ballooning envelope from TJLF's WIDTH_SPECTRUM
    env_t = [exp(-θ_g[k]^2 / (2 * θw_ky[j]^2)) for k in 1:ntheta, j in 1:nky]

    phi_amp = sqrt.(max.(dropdims(sum(phi2; dims=3); dims=3), 0))
    @inbounds for ikx in 1:nkx, j in 1:nky
        if abs(fs_kx[ikx]) > k_nyq_rs || fs_ky[j] > ky_cap
            phi_amp[ikx, j] = 0.0
        end
    end

    phi_rt = zeros(Float64, nr, ntheta)
    @inbounds for k in 1:ntheta
        θ   = θ_g[k]
        pθ_const = -r0_rs * θ
        kx_tilt  = -shat  * θ
        for j in 1:nky
            env = env_t[k, j]
            env == 0 && continue
            kyj = fs_ky[j]
            kyj > ky_cap && continue
            pθ  = kyj * pθ_const
            Δkx = kyj * kx_tilt
            for ikx in 1:nkx
                amp = phi_amp[ikx, j]
                amp == 0 && continue
                ktot = fs_kx[ikx] + Δkx
                ψ    = phases[ikx, j]
                for ir in 1:nr
                    phi_rt[ir, k] += amp * cos(ktot * x_rs[ir] + pθ + ψ) * env
                end
            end
        end
    end
    phi_rt ./= max(maximum(abs, phi_rt), eps(Float64))

    return (r_g, Interpolations.extrapolate(
        Interpolations.interpolate((r_g, θ_g), phi_rt, Gridded(Linear())),
        (Flat(), Periodic())))
end

# Per-band raster pitch -> k_nyq in ρ_s units.  Use the outer-band bounding
# box (biggest in R, Z) so the estimate is a safe upper bound everywhere.
Δpix_a_glob = max((Rmax_bb_est - Rmin_bb_est) / nR_pix_multi,
                  (Zmax_bb_est - Zmin_bb_est) / nZ_pix_multi)

Threads.@threads for i in eachindex(bands)
    b     = bands[i]
    inp   = b.input
    fs_i  = b.fs
    rs_a  = b.rho_s_over_a
    rmin0 = inp.RMIN_LOC
    q0    = abs(inp.Q_LOC)
    shat  = inp.Q_PRIME_LOC * rmin0^2 / q0^2
    θw_ky = collect(Float64, inp.WIDTH_SPECTRUM)

    r_center_lo = i == 1              ? rmin_edges[1]        : rmin_centers[i - 1]
    r_center_hi = i == length(bands)  ? rmin_edges[end]      : rmin_centers[i + 1]
    r_lo  = max(r_center_lo - 0.01, 1e-3)
    r_hi  = min(r_center_hi + 0.01, 0.999)

    k_nyq_rs = π * rs_a / Δpix_a_glob
    ky_cap_i = min(ky_cutoff_multi, k_nyq_rs)

    _, band_itps[i] = _build_band_itp(
        Float64(rmin0), Float64(q0), Float64(shat), θw_ky, Float64(rs_a),
        Float64(r_lo), Float64(r_hi),
        collect(Float64, fs_i.kx), collect(Float64, fs_i.ky),
        Array{Float64,3}(fs_i.phi2), Matrix{Float64}(band_phases[i]),
        collect(Float64, θ_g), Int(nr_band),
        Float64(k_nyq_rs), Float64(ky_cap_i))
end

# --- Global (R, Z) ↔ (r, θ) with radially-varying Miller coefficients ---
# Linear interpolants of (R₀, Z₀, κ, δ, ζ) over the bands' RMIN_LOC knots.
# Inversion uses 2-D damped Newton on the resulting r-dependent map so that
# the global frame includes Shafranov shift and shape gradients.
ref = bands[fld(length(bands) + 1, 2)].input  # central band (kept for downstream diagnostics)
let
    knots = sort([b.input.RMIN_LOC for b in bands])
    perm  = sortperm([b.input.RMIN_LOC for b in bands])
    Rmaj_k = [bands[p].input.RMAJ_LOC  for p in perm]
    Zmaj_k = [bands[p].input.ZMAJ_LOC  for p in perm]
    Kap_k  = [bands[p].input.KAPPA_LOC for p in perm]
    Del_k  = [bands[p].input.DELTA_LOC for p in perm]
    Zet_k  = [bands[p].input.ZETA_LOC  for p in perm]
    global _knots   = collect(Float64, knots)
    global _Rmaj_k  = collect(Float64, Rmaj_k)
    global _Zmaj_k  = collect(Float64, Zmaj_k)
    global _Kap_k   = collect(Float64, Kap_k)
    global _Del_k   = collect(Float64, Del_k)
    global _Zet_k   = collect(Float64, Zet_k)
end

# Linear extrapolation outside the knot range (Line() boundary condition mimic).
function _lin_eval(knots::Vector{Float64}, vals::Vector{Float64}, r::Float64)
    n = length(knots)
    if n == 1
        return (vals[1], 0.0)
    end
    if r <= knots[1]
        m = (vals[2] - vals[1]) / (knots[2] - knots[1])
        return (vals[1] + m * (r - knots[1]), m)
    elseif r >= knots[end]
        m = (vals[end] - vals[end-1]) / (knots[end] - knots[end-1])
        return (vals[end] + m * (r - knots[end]), m)
    else
        i = searchsortedlast(knots, r)
        i = clamp(i, 1, n - 1)
        m = (vals[i+1] - vals[i]) / (knots[i+1] - knots[i])
        return (vals[i] + m * (r - knots[i]), m)
    end
end

function miller_RZ_ref(r, θ)
    R0r, _ = _lin_eval(_knots, _Rmaj_k, Float64(r))
    Z0r, _ = _lin_eval(_knots, _Zmaj_k, Float64(r))
    κr,  _ = _lin_eval(_knots, _Kap_k,  Float64(r))
    δr,  _ = _lin_eval(_knots, _Del_k,  Float64(r))
    ζr,  _ = _lin_eval(_knots, _Zet_k,  Float64(r))
    δr = clamp(δr, -0.999, 0.999)
    return (R0r + r * cos(θ + asin(δr)*sin(θ) - ζr*sin(2θ)),
            Z0r + κr * r * sin(θ))
end

function invert_miller_ref(Rq, Zq; iters = 24, tol = 1e-10)
    # initial guess: invert with central-band coefficients (concentric Miller)
    ref_idx = fld(length(_knots) + 1, 2)
    rmaj0   = _Rmaj_k[ref_idx]
    zmaj0   = _Zmaj_k[ref_idx]
    κ0      = _Kap_k[ref_idx]
    δ0      = _Del_k[ref_idx]
    ζ0      = _Zet_k[ref_idx]
    s_r = Rq - rmaj0
    s_z = (Zq - zmaj0) / κ0
    θ   = atan(s_z, s_r)
    asd = asin(δ0)
    for _ in 1:6
        a  = θ + asd*sin(θ) - ζ0*sin(2θ)
        ap = 1 + asd*cos(θ) - 2*ζ0*cos(2θ)
        f  = s_z*cos(a) - s_r*sin(θ)
        df = -s_z*sin(a)*ap - s_r*cos(θ)
        abs(df) < 1e-14 && break
        Δ = f / df
        abs(Δ) > 0.5 && (Δ = 0.5 * sign(Δ))
        θ -= Δ
        abs(Δ) < 1e-8 && break
    end
    a0 = θ + asd*sin(θ) - ζ0*sin(2θ)
    r  = (abs(cos(a0)) >= abs(sin(θ))) ? s_r/cos(a0) : s_z/sin(θ)
    r < 0 && (r = -r; θ += π)

    for _ in 1:iters
        R0r, dR0 = _lin_eval(_knots, _Rmaj_k, Float64(r))
        Z0r, dZ0 = _lin_eval(_knots, _Zmaj_k, Float64(r))
        κr,  dκr = _lin_eval(_knots, _Kap_k,  Float64(r))
        δr,  dδr = _lin_eval(_knots, _Del_k,  Float64(r))
        ζr,  dζr = _lin_eval(_knots, _Zet_k,  Float64(r))
        δr = clamp(δr, -0.999, 0.999)
        cdr = sqrt(max(1 - δr*δr, 1e-12))
        asd  = asin(δr)
        dasd = dδr / cdr
        argR    = θ + asd*sin(θ) - ζr*sin(2θ)
        dargRdr = dasd*sin(θ) - dζr*sin(2θ)
        dargRdθ = 1 + asd*cos(θ) - 2*ζr*cos(2θ)
        cR, sR = cos(argR), sin(argR)
        F1 = R0r + r*cR - Rq
        F2 = Z0r + κr*r*sin(θ) - Zq
        J11 = dR0 + cR - r*sR*dargRdr
        J12 = -r*sR*dargRdθ
        J21 = dZ0 + (κr + dκr*r)*sin(θ)
        J22 = κr*r*cos(θ)
        det = J11*J22 - J12*J21
        abs(det) < 1e-14 && break
        δr_step = (J22*F1 - J12*F2) / det
        δθ_step = (J11*F2 - J21*F1) / det
        s = max(abs(δr_step) / 0.1, abs(δθ_step) / 0.5, 1.0)
        r -= δr_step / s
        θ -= δθ_step / s
        max(abs(δr_step), abs(δθ_step)) < tol && break
    end
    if r < 0
        r = -r
        θ += π
    end
    return r, mod(θ + π, 2π) - π
end

# --- Rasterise ---
r_inner_plasma = rmin_edges[1]
r_outer_plasma = rmin_edges[end]
θ_bb  = collect(range(-π, π; length=ntheta_band))
Rmin_bb = minimum(miller_RZ_ref(r_outer_plasma, θ)[1] for θ in θ_bb) - 0.03
Rmax_bb = maximum(miller_RZ_ref(r_outer_plasma, θ)[1] for θ in θ_bb) + 0.03
Zmin_bb = minimum(miller_RZ_ref(r_outer_plasma, θ)[2] for θ in θ_bb) - 0.03
Zmax_bb = maximum(miller_RZ_ref(r_outer_plasma, θ)[2] for θ in θ_bb) + 0.03
Rpix = collect(range(Rmin_bb, Rmax_bb; length=nR_pix_multi))
Zpix = collect(range(Zmin_bb, Zmax_bb; length=nZ_pix_multi))
phi_raster_multi = fill(NaN, nR_pix_multi, nZ_pix_multi)

# Variance-preserving cosine blend between the two bands that bracket rmin:
#   w_lo = cos(π t / 2),  w_hi = sin(π t / 2),  so w_lo^2 + w_hi^2 = 1
# — independent random-phase realisations keep uniform variance across the seam.
function _raster_multi!(phi_raster_multi::Matrix{Float64},
        Rpix::Vector{Float64}, Zpix::Vector{Float64},
        invert_miller_ref, band_itps::Vector,
        rmin_centers::Vector{Float64},
        r_inner::Float64, r_outer::Float64)
    nbands = length(band_itps)
    nR, nZ = length(Rpix), length(Zpix)
    Threads.@threads for jp in 1:nZ
        @inbounds for ip in 1:nR
            r, θ = invert_miller_ref(Rpix[ip], Zpix[jp])
            (r_inner ≤ r ≤ r_outer) || continue
            if r ≤ rmin_centers[1]
                phi_raster_multi[ip, jp] = band_itps[1](r, θ)
            elseif r ≥ rmin_centers[end]
                phi_raster_multi[ip, jp] = band_itps[nbands](r, θ)
            else
                i_lo = searchsortedlast(rmin_centers, r)
                i_hi = i_lo + 1
                t    = (r - rmin_centers[i_lo]) / (rmin_centers[i_hi] - rmin_centers[i_lo])
                w_lo = cos(π * t / 2)
                w_hi = sin(π * t / 2)
                phi_raster_multi[ip, jp] = w_lo * band_itps[i_lo](r, θ) +
                                           w_hi * band_itps[i_hi](r, θ)
            end
        end
    end
    return phi_raster_multi
end

# --- Time-independent raster geometry precompute ---------------------------
# Every pixel's (r, θ), the band index that owns it, and the cosine-blend
# weight depend only on the pixel grid + Miller geometry — NOT on time. The
# movie cell calls `_precompute_raster` once and reuses the resulting tables
# at every frame via `_raster_multi_fast!`, eliminating the per-frame Newton
# inversion and the per-frame `searchsortedlast`.
#
# pix_ilo encoding:
#   0  = pixel outside [r_inner, r_outer]   (skip)
#  -1  = r ≤ rmin_centers[1]                (use band 1 only)
#  -2  = r ≥ rmin_centers[end]              (use band end only)
#   k  = blend bands k and k+1 with weights (cos(πt/2), sin(πt/2))
function _precompute_raster(
        Rpix::Vector{Float64}, Zpix::Vector{Float64},
        invert_miller_ref,
        rmin_centers::Vector{Float64},
        r_inner::Float64, r_outer::Float64)
    nR, nZ = length(Rpix), length(Zpix)
    pix_r   = fill(NaN, nR, nZ)
    pix_θ   = fill(NaN, nR, nZ)
    pix_ilo = fill(Int32(0), nR, nZ)
    pix_wlo = fill(0.0, nR, nZ)
    Threads.@threads for jp in 1:nZ
        @inbounds for ip in 1:nR
            r, θ = invert_miller_ref(Rpix[ip], Zpix[jp])
            (r_inner ≤ r ≤ r_outer) || continue
            pix_r[ip, jp] = r
            pix_θ[ip, jp] = θ
            if r ≤ rmin_centers[1]
                pix_ilo[ip, jp] = Int32(-1)
            elseif r ≥ rmin_centers[end]
                pix_ilo[ip, jp] = Int32(-2)
            else
                i_lo = searchsortedlast(rmin_centers, r)
                t    = (r - rmin_centers[i_lo]) / (rmin_centers[i_lo+1] - rmin_centers[i_lo])
                pix_ilo[ip, jp] = Int32(i_lo)
                pix_wlo[ip, jp] = cos(π * t / 2)
            end
        end
    end
    return (; pix_r, pix_θ, pix_ilo, pix_wlo)
end

function _raster_multi_fast!(phi_raster::Matrix{Float64},
        pix_r::Matrix{Float64}, pix_θ::Matrix{Float64},
        pix_ilo::Matrix{Int32}, pix_wlo::Matrix{Float64},
        band_itps::Vector, nbands::Int)
    nR, nZ = size(phi_raster)
    Threads.@threads for jp in 1:nZ
        @inbounds for ip in 1:nR
            ilo = pix_ilo[ip, jp]
            ilo == 0 && continue
            r = pix_r[ip, jp]; θ = pix_θ[ip, jp]
            if ilo == -1
                phi_raster[ip, jp] = band_itps[1](r, θ)
            elseif ilo == -2
                phi_raster[ip, jp] = band_itps[nbands](r, θ)
            else
                wlo = pix_wlo[ip, jp]
                whi = sqrt(max(1.0 - wlo*wlo, 0.0))
                phi_raster[ip, jp] = wlo * band_itps[ilo](r, θ) + whi * band_itps[ilo+1](r, θ)
            end
        end
    end
    return phi_raster
end

_raster_multi!(phi_raster_multi,
    collect(Float64, Rpix), collect(Float64, Zpix),
    invert_miller_ref, band_itps,
    collect(Float64, rmin_centers),
    Float64(r_inner_plasma), Float64(r_outer_plasma))
# --- torcut diagnostic ------------------------------------------------------
### torcut diagnostic
filled = count(!isnan, phi_raster_multi)
total  = length(phi_raster_multi)
println("─"^60)
println("torcut raster diagnostic")
println("  rmin_centers = $(round.(rmin_centers; digits=3))")
println("  rmin_edges   = $(round.(rmin_edges;   digits=3))")
println("  r_inner / r_outer = $(round(r_inner_plasma; digits=3)) / $(round(r_outer_plasma; digits=3))")
println("  band_itps assigned: $(all(i -> isassigned(band_itps, i), 1:length(band_itps))) ($(count(i -> isassigned(band_itps, i), 1:length(band_itps)))/$(length(band_itps)))")
println("  raster filled: $filled / $total  ($(round(100*filled/total; digits=1)) %)")
println("  θw per-ky (rad): min/max/mean per band:")
for (i, b) in enumerate(bands)
    w = b.input.WIDTH_SPECTRUM
    println("    band $i (ρ=$(round(b.ρ; digits=3))): min=$(round(minimum(w); digits=3)), max=$(round(maximum(w); digits=3)), mean=$(round(sum(w)/length(w); digits=3))")
end
# sample a few (R, Z) → (r, θ) inversions so we can see the mapping
for (Rs, Zs) in ((ref.RMAJ_LOC, ref.ZMAJ_LOC),
                 (ref.RMAJ_LOC + 0.2, ref.ZMAJ_LOC),
                 (ref.RMAJ_LOC + rmin_centers[end], ref.ZMAJ_LOC))
    r_s, θ_s = invert_miller_ref(Rs, Zs)
    println("  (R=$(round(Rs;digits=3)), Z=$(round(Zs;digits=3)))  →  r=$(round(r_s;digits=3)), θ=$(round(θ_s;digits=3))")
end
println("─"^60)
phi_raster_multi   # keep the cell's return value as the raster


In [ ]:
# --- Plot ---
using Plots.PlotMeasures
using LaTeXStrings
vmax_m = maximum(abs, filter(!isnan, phi_raster_multi))
ρ_lo = round(minimum(rho_list_multi), digits=2)
ρ_hi = round(maximum(rho_list_multi), digits=2)
p_multi = heatmap(Rpix, Zpix, phi_raster_multi';
    c = cgrad(:RdBu_10, rev=true), clims = (-vmax_m, vmax_m),
    aspect_ratio = :equal, colorbar = false, legend = false, framestyle = :box,
    xlabel = L"R / a", ylabel = L"Z / a", size = (540, 760),
    left_margin = 4mm, right_margin = 4mm, bottom_margin = 4mm, top_margin = 4mm,
    fontfamily = "Computer Modern",
    titlefontfamily = "Computer Modern",
    guidefontfamily = "Computer Modern",
    tickfontfamily  = "Computer Modern",
    title = L"|\varphi|\ \ (\rho \in [%$ρ_lo,\,%$ρ_hi])")

# overlay: each band's central ρ-surface (dotted) + inner/outer envelopes (solid)
for b in bands
    Rl = [miller_RZ_ref(b.input.RMIN_LOC, θ)[1] for θ in θ_bb]
    Zl = [miller_RZ_ref(b.input.RMIN_LOC, θ)[2] for θ in θ_bb]
    plot!(p_multi, Rl, Zl; color = :black, lw = 0.4, ls = :dot)
end
R_out_line = [miller_RZ_ref(r_outer_plasma, θ)[1] for θ in θ_bb]
Z_out_line = [miller_RZ_ref(r_outer_plasma, θ)[2] for θ in θ_bb]
R_in_line  = [miller_RZ_ref(r_inner_plasma, θ)[1] for θ in θ_bb]
Z_in_line  = [miller_RZ_ref(r_inner_plasma, θ)[2] for θ in θ_bb]
plot!(p_multi, R_out_line, Z_out_line; color = :black, lw = 1.4)
plot!(p_multi, R_in_line,  Z_in_line;  color = :black, lw = 1.4)
p_multi
savefig(p_multi, "tjlf_torcut_multirho.pdf")
p_multi


In [ ]:
# ============================================================================
# Time-evolution movie of the single-ρ torcut
# ============================================================================
# Animates the synthetic φ realization at ρ_plot in real time. The phase of
# every (kx, ky) Fourier component is advanced by −ω_ky·t where ω_ky is the
# dominant-mode frequency from TJLF. The sign of ω is preserved: positive
# (electron-diamagnetic) and negative (ion-diamagnetic) modes propagate in
# opposite poloidal directions, so mixed-sign turbulence (low-ky ITG vs high-ky
# TEM/ETG) animates as counter-streaming patterns.
import TJLF
using Printf
using Plots.PlotMeasures
using LaTeXStrings

# --- knobs ---
n_frames    = 40
t_periods   = 50.0                              # span this many dominant-mode periods
mp4_fps     = 5                                 # MP4 playback rate (input + output via -r)
gif_fps     = 10                                # GIF playback rate (set via filter)
out_dir_s   = "./torcut_movie_single"
gif_width   = 480

# --- 1) Per-ky dominant-mode (γ, ω) from cached eigenvalues ---
γω        = tjlf_eigval                          # populated by cell 29 (cache or fresh run)
nm_, nky_, _ = size(γω)
m_dom = [argmax(γω[:, j, 1]) for j in 1:nky_]
ω_ky  = Float64[γω[m_dom[j], j, 2] for j in 1:nky_]   # SIGNED, c_s/a units
γ_ky  = Float64[γω[m_dom[j], j, 1] for j in 1:nky_]

# --- 2) Real-time normalization at ρ_plot: c_s/a [s⁻¹] ---
cs_over_a = IMAS.interp1d(csa_rho, csa_prof)(rho_plot)

# --- 3) Auto-scaled window: span t_periods of the most-ENERGETIC ky's mode ---
# Use the ky that contributes most to the visible field (max phinorm),
# not max|ω| (which is dominated by high-ky outliers and would make the bulk
# turbulence barely move between frames).
phinorm_per_ky = vec(sum(fs.phi2_ky; dims=2))                    # (nky,) summed over modes
contributing   = findall(j -> γ_ky[j] > 0 && phinorm_per_ky[j] > 0, 1:nky_)
@assert !isempty(contributing) "no ky has γ > 0 AND phinorm > 0 — nothing to animate"
j_dom          = contributing[argmax(phinorm_per_ky[contributing])]
ω_dom_norm     = ω_ky[j_dom]                                     # SIGNED, c_s/a
ω_dom_phys     = abs(ω_dom_norm) * cs_over_a                     # [rad/s]
T_dom_phys     = 2π / max(ω_dom_phys, 1e-30)                     # [s]
t_window_s     = t_periods * T_dom_phys
t_norms        = collect(range(0.0, t_window_s * cs_over_a; length=n_frames))
Δt_norm        = n_frames > 1 ? t_norms[2] - t_norms[1] : 0.0

n_e = count(>(0), ω_ky); n_i = count(<(0), ω_ky); n_z = count(==(0), ω_ky)
@info("Single-ρ animation setup",
    rho_plot, n_frames, t_periods,
    dominant_ky=fs.ky[j_dom], dominant_phinorm=phinorm_per_ky[j_dom],
    dominant_ω_norm=ω_dom_norm, dominant_ω_kHz=ω_dom_phys/(2π)*1e-3,
    t_window_us=t_window_s*1e6, cs_over_a,
    n_electron_modes=n_e, n_ion_modes=n_i, n_zero=n_z)
println("─"^60)
println("Per-ky dominant mode (signs: + electron, − ion); * = scaling reference)")
for j in 1:nky_
    sgn   = ω_ky[j] > 0 ? "e" : (ω_ky[j] < 0 ? "i" : "0")
    star  = j == j_dom ? " *" : "  "
    Δψ_deg = ω_ky[j] * Δt_norm * 180/π
    @printf("  ky·ρ_s=%6.3f   γ=% .3e   ω=% .3e   [%s]   Δψ/frame=% 7.1f°%s\n",
            fs.ky[j], γ_ky[j], ω_ky[j], sgn, Δψ_deg, star)
end
println("─"^60)

# --- 4) Deterministic base phases (matches cell 31's t=0 frame) ---
Random.seed!(123)
base_phases_s = 2π .* rand(length(fs.kx), length(fs.ky))

# --- 5) Stable color scale: peak amplitude of the t=0 reconstruction ---
phi_t0_peak = let
    phi_rt0 = zeros(Float64, nr_mesh, ntheta_mesh)
    _reconstruct_phi_rt!(phi_rt0, collect(Float64, theta_grid), Matrix{Float64}(env_θky),
        collect(Float64, fs.kx), collect(Float64, fs.ky),
        Matrix{Float64}(phi_amp), Matrix{Float64}(base_phases_s),
        collect(Float64, x_rs), Float64(r0_rs), Float64(shat),
        collect(Int, ky_use), Int(nr_mesh), Int(ntheta_mesh))
    max(maximum(abs, phi_rt0), eps(Float64))
end

# --- 6) Bounding-box pixels (re-derive locally so the cell is order-robust) ---
Rmin_bb_s = minimum(miller_RZ(r_out, θ)[1] for θ in theta_grid) - 0.03
Rmax_bb_s = maximum(miller_RZ(r_out, θ)[1] for θ in theta_grid) + 0.03
Zmin_bb_s = minimum(miller_RZ(r_out, θ)[2] for θ in theta_grid) - 0.03
Zmax_bb_s = maximum(miller_RZ(r_out, θ)[2] for θ in theta_grid) + 0.03
Rpix_s = collect(range(Rmin_bb_s, Rmax_bb_s; length=nR_pix))
Zpix_s = collect(range(Zmin_bb_s, Zmax_bb_s; length=nZ_pix))

# --- 7) Frame loop ---
mkpath(out_dir_s)
for (i, t_norm) in enumerate(t_norms)
    phases_t = base_phases_s .- reshape(ω_ky, 1, :) .* t_norm   # signed ω

    phi_rt_t = zeros(Float64, nr_mesh, ntheta_mesh)
    _reconstruct_phi_rt!(phi_rt_t, collect(Float64, theta_grid), Matrix{Float64}(env_θky),
        collect(Float64, fs.kx), collect(Float64, fs.ky),
        Matrix{Float64}(phi_amp), Matrix{Float64}(phases_t),
        collect(Float64, x_rs), Float64(r0_rs), Float64(shat),
        collect(Int, ky_use), Int(nr_mesh), Int(ntheta_mesh))
    phi_rt_t ./= phi_t0_peak

    itp_phi_t = Interpolations.extrapolate(
        Interpolations.interpolate((r_grid, theta_grid), phi_rt_t, Gridded(Linear())),
        (Flat(), Periodic()))

    phi_raster_t = fill(NaN, nR_pix, nZ_pix)
    _raster_torcut!(phi_raster_t, collect(Float64, Rpix_s), collect(Float64, Zpix_s),
        itp_phi_t, invert_miller, Float64(r_in), Float64(r_out))

    t_us  = t_norm / cs_over_a * 1e6
    ρ_str = @sprintf("%.2f", rho_plot)
    sat_n = Int(input_tjlf.SAT_RULE)
    t_str = @sprintf("%.1f", t_us)
    p_t = heatmap(Rpix_s, Zpix_s, phi_raster_t';
        c = cgrad(:RdBu_10, rev=true), clims = (-1.0, 1.0),
        aspect_ratio = :equal, colorbar = false, legend = false, framestyle = :box,
        xlabel = L"R / a", ylabel = L"Z / a", size = (540, 760),
        left_margin = 4mm, right_margin = 4mm, bottom_margin = 4mm, top_margin = 4mm,
        fontfamily = "Computer Modern",
        titlefontfamily = "Computer Modern",
        guidefontfamily = "Computer Modern",
        tickfontfamily  = "Computer Modern",
        title = L"|\varphi|\ \ (\rho=%$ρ_str,\ \mathrm{SAT}%$sat_n)\quad t=%$t_str\,\mu s\quad [%$i/%$n_frames]")
    plot!(p_t, [miller_RZ(r_in,  θ)[1] for θ in theta_grid],
              [miller_RZ(r_in,  θ)[2] for θ in theta_grid]; color=:black, lw=1.4)
    plot!(p_t, [miller_RZ(r_out, θ)[1] for θ in theta_grid],
              [miller_RZ(r_out, θ)[2] for θ in theta_grid]; color=:black, lw=1.4)
    plot!(p_t, [miller_RZ(rmin0, θ)[1] for θ in theta_grid],
              [miller_RZ(rmin0, θ)[2] for θ in theta_grid]; color=:black, lw=0.8, ls=:dash)
    savefig(p_t, joinpath(out_dir_s, @sprintf("frame_%04d.png", i)))
end
println("✔ wrote $n_frames PNGs to $out_dir_s")

# --- 8) FFmpeg → MP4 + GIF ---
# Resolve ffmpeg binary: PATH first, then common conda env locations
ffmpeg_bin = let p = Sys.which("ffmpeg")
    if p !== nothing
        p
    else
        candidates = [
            joinpath(homedir(), "opt", "anaconda3", "envs", "ffmpeg", "bin", "ffmpeg"),
            joinpath(homedir(), "anaconda3",       "envs", "ffmpeg", "bin", "ffmpeg"),
            joinpath(homedir(), "miniconda3",      "envs", "ffmpeg", "bin", "ffmpeg"),
            joinpath(homedir(), "miniforge3",      "envs", "ffmpeg", "bin", "ffmpeg"),
        ]
        idx = findfirst(isfile, candidates)
        idx === nothing ? nothing : candidates[idx]
    end
end

if ffmpeg_bin !== nothing
    @info "Using ffmpeg" ffmpeg_bin
    mp4_s     = joinpath(out_dir_s, "movie.mp4")
    gif_s     = joinpath(out_dir_s, "movie.gif")
    pattern_s = joinpath(out_dir_s, "frame_%04d.png")
    try
        Base.run(`$ffmpeg_bin -y -loglevel error -framerate $mp4_fps -i $pattern_s -c:v libx264 -pix_fmt yuv420p -r $mp4_fps -crf 18 $mp4_s`)
        @info "MP4 written" mp4_s
        Base.run(`$ffmpeg_bin -y -loglevel error -framerate $mp4_fps -i $pattern_s -filter_complex "fps=$gif_fps,scale=$gif_width:-1:flags=lanczos,split[s0][s1];[s0]palettegen[p];[s1][p]paletteuse" -r $gif_fps $gif_s`)
        @info "GIF written" gif_s
        display("image/gif", read(gif_s))
    catch err
        @warn "ffmpeg failed" err
    end
else
    @warn "ffmpeg not found in PATH and no conda envs/ffmpeg/bin/ffmpeg — keeping the $n_frames PNGs in $out_dir_s"
end


In [ ]:
# ============================================================================
# Time-evolution movie of the multi-band torcut  (band-streamed)
# ============================================================================
# Per-band per-ky dominant-mode ω drives time evolution. Each band uses its
# own local c_s/a (Te varies with ρ), so for a real time t [s] band k has
# normalized time t · cs_over_a(ρ_k). Phase shift is signed (+ω electron,
# −ω ion direction); see the single-ρ cell for the convention.
#
# Band-streamed memory plan
# -------------------------
# `ntheta_band` is auto-sized to Nyquist of the smallest-scale ky·ρ_s mode and
# can reach 50k+. The per-band complex-amplitude tables Ar/Ai are
# (nky × nr_band × ntheta_band) doubles → 1+ GB each. Keeping all bands'
# tables resident at once peaks at tens of GB and shoves the kernel into
# swap. Instead we stream:
#
#   for each band k:
#     allocate Ar_b, Ai_b           (peak memory: ONE band's worth)
#     _precompute_band_complex!     (cos/sin sum over kx,ky,r,θ — once)
#     ── vmax pre-pass: scan n_frames `_recon_band_fast!` calls,
#        track vmax_per_band[k]
#     ── production pass: for each frame, `_recon_band_fast!` →
#        per-band scaled interp → accumulate into frame_rasters[i] at
#        pixels assigned to band k by the raster blend tables
#     Ar_b = Ai_b = nothing  +  GC.gc()    → next band reuses the slot
#     @info "band k done"                  → user sees progress
#
# Plotting then becomes a tiny pass: just `heatmap` + `savefig` over the
# pre-rasterized `frame_rasters`. PNGs no longer wait on per-band precomputes,
# and total peak memory drops from O(n_bands · GB) to O(1 · GB).
import TJLF
using Printf
using Plots.PlotMeasures
using LaTeXStrings
using Statistics

# --- knobs ---
n_frames_m        = 80
t_periods_m       = 50.0
mp4_fps_m         = 5                               # MP4 playback rate
gif_fps_m         = 10                              # GIF playback rate
out_dir_m         = "./torcut_movie_multi"
gif_width_m       = 480
# Time-axis selection: which band's dominant ω sets the window?
#   0.0 = slowest band   (long window; core aliases between frames)
#   0.5 = median band    (default; balances core + edge visibility)
#   1.0 = fastest band   (short window; edge barely moves)
master_ω_quantile = 1.0
# Color-scale normalization across the movie:
#   :per_band — each band normalized by its own max|ϕ| over time
#               (every radius stays visible; cross-band amplitude lost)
#   :global   — single denominator across all bands and frames
#               (physically truthful; quiet bands stay visually quiet)
normalize_mode    = :per_band

# --- 1) Per-band dominant-mode (γ, ω) from cache + cs_over_a(ρ_band) ---
band_ωky = Vector{Vector{Float64}}(undef, length(bands))
band_γky = Vector{Vector{Float64}}(undef, length(bands))
band_csa = Float64[]
csa_itp  = IMAS.interp1d(csa_rho, csa_prof)
for (i, b) in enumerate(bands)
    γω  = band_eigenvalue[i]                          # (nm, nky_b, 2)
    _, nky_b, _ = size(γω)
    md  = [argmax(γω[:, j, 1]) for j in 1:nky_b]
    band_ωky[i] = Float64[γω[md[j], j, 2] for j in 1:nky_b]
    band_γky[i] = Float64[γω[md[j], j, 1] for j in 1:nky_b]
    push!(band_csa, csa_itp(b.ρ))
end

# --- 2) Auto-scaled real-time window using the most-energetic mode per band ---
band_dom_ω_phys = Float64[]
band_dom_ky     = Float64[]
band_dom_jdom   = Int[]
for (i, b) in enumerate(bands)
    pn = vec(sum(b.fs.phi2_ky; dims=2))
    γ  = band_γky[i]
    ω  = band_ωky[i]
    contrib = findall(j -> γ[j] > 0 && pn[j] > 0, eachindex(pn))
    if isempty(contrib)
        push!(band_dom_ω_phys, 0.0); push!(band_dom_ky, NaN); push!(band_dom_jdom, 0)
        continue
    end
    j_dom_b = contrib[argmax(pn[contrib])]
    push!(band_dom_jdom, j_dom_b)
    push!(band_dom_ky, b.fs.ky[j_dom_b])
    push!(band_dom_ω_phys, abs(ω[j_dom_b]) * band_csa[i])
end
ω_master_phys = let nz = filter(>(0), band_dom_ω_phys)
    isempty(nz) ? 1e-30 : quantile(nz, master_ω_quantile)   # `master_ω_quantile`-th band sets the window
end
T_dom_phys_m  = 2π / max(ω_master_phys, 1e-30)
t_window_s_m  = t_periods_m * T_dom_phys_m
t_seconds_m   = collect(range(0.0, t_window_s_m; length=n_frames_m))
Δt_s          = n_frames_m > 1 ? t_seconds_m[2] - t_seconds_m[1] : 0.0

println("─"^60)
println("Multi-band animation setup")
@printf("  bands               = %d\n", length(bands))
@printf("  n_frames            = %d\n", n_frames_m)
@printf("  t_periods           = %.1f\n", t_periods_m)
@printf("  master ω (quantile=%.2f)   = %.2f kHz\n", master_ω_quantile, ω_master_phys / (2π) * 1e-3)
@printf("  t_window            = %.1f µs\n", t_window_s_m * 1e6)
println("  per-band dominant-mode summary (sign: + electron, − ion; * if it sets the window):")
for (i, b) in enumerate(bands)
    ne = count(>(0), band_ωky[i]); ni = count(<(0), band_ωky[i])
    j_dom_b = band_dom_jdom[i]
    star    = (band_dom_ω_phys[i] == ω_master_phys) ? " *" : "  "
    Δψ_deg  = band_dom_jdom[i] == 0 ? 0.0 : band_ωky[i][j_dom_b] * (Δt_s * band_csa[i]) * 180/π
    @printf("    band %d (ρ=%.3f): cs/a=%.2e s⁻¹, n_e=%d, n_i=%d, ky_dom=%.3f, ω_dom=% .3e (kHz=% .2f), Δψ/frame=% 7.1f°%s\n",
        i, b.ρ, band_csa[i], ne, ni,
        isnan(band_dom_ky[i]) ? NaN : band_dom_ky[i],
        j_dom_b == 0 ? 0.0 : band_ωky[i][j_dom_b],
        band_dom_ω_phys[i] / (2π) * 1e-3,
        Δψ_deg, star)
end
println("─"^60)

# --- 3) Bounding-box pixels (re-derive locally) ---
θ_bb_a    = collect(range(-π, π; length=ntheta_band))
Rmin_bb_m = minimum(miller_RZ_ref(r_outer_plasma, θ)[1] for θ in θ_bb_a) - 0.03
Rmax_bb_m = maximum(miller_RZ_ref(r_outer_plasma, θ)[1] for θ in θ_bb_a) + 0.03
Zmin_bb_m = minimum(miller_RZ_ref(r_outer_plasma, θ)[2] for θ in θ_bb_a) - 0.03
Zmax_bb_m = maximum(miller_RZ_ref(r_outer_plasma, θ)[2] for θ in θ_bb_a) + 0.03
Rpix_m = collect(range(Rmin_bb_m, Rmax_bb_m; length=nR_pix_multi))
Zpix_m = collect(range(Zmin_bb_m, Zmax_bb_m; length=nZ_pix_multi))

# Static contour cache (geometry-only; precomputed once, reused every frame).
band_contour_R = [[miller_RZ_ref(b.input.RMIN_LOC, θ)[1] for θ in θ_bb_a] for b in bands]
band_contour_Z = [[miller_RZ_ref(b.input.RMIN_LOC, θ)[2] for θ in θ_bb_a] for b in bands]
outer_R        =  [miller_RZ_ref(r_outer_plasma, θ)[1] for θ in θ_bb_a]
outer_Z        =  [miller_RZ_ref(r_outer_plasma, θ)[2] for θ in θ_bb_a]
inner_R        =  [miller_RZ_ref(r_inner_plasma, θ)[1] for θ in θ_bb_a]
inner_Z        =  [miller_RZ_ref(r_inner_plasma, θ)[2] for θ in θ_bb_a]

# One-shot raster geometry (pix_r, pix_θ, pix_ilo, pix_wlo).
raster_geo_m = _precompute_raster(
    collect(Float64, Rpix_m), collect(Float64, Zpix_m),
    invert_miller_ref, collect(Float64, rmin_centers),
    Float64(r_inner_plasma), Float64(r_outer_plasma))
@info "raster geometry precomputed" active_pixels=count(!=(0), raster_geo_m.pix_ilo) total_pixels=length(raster_geo_m.pix_ilo)

# --- 4) Per-band active-pixel SoA tables (key to band-streaming) ---
# For each band k, list every (ip, jp) it contributes to AND the weight at
# which it does so. A pixel can be touched by up to 2 adjacent bands; the
# cosine-blend weights pix_wlo (and √(1−pix_wlo²) for the high band) come
# straight from the rasterizer convention. Packing per-band SoA arrays gives
# `_accumulate_band!` a tight, threaded inner loop.
function _build_band_pixel_tables(raster_geo, nbands::Int)
    pix_ilo = raster_geo.pix_ilo
    pix_wlo = raster_geo.pix_wlo
    pix_r   = raster_geo.pix_r
    pix_θ   = raster_geo.pix_θ
    nR, nZ  = size(pix_ilo)

    counts = zeros(Int, nbands)
    @inbounds for jp in 1:nZ, ip in 1:nR
        ilo = pix_ilo[ip, jp]
        ilo == 0 && continue
        if ilo == -1
            counts[1] += 1
        elseif ilo == -2
            counts[nbands] += 1
        else
            counts[ilo]     += 1
            counts[ilo + 1] += 1
        end
    end

    tables = [(
        ip = Vector{Int32}(undef, counts[k]),
        jp = Vector{Int32}(undef, counts[k]),
        r  = Vector{Float64}(undef, counts[k]),
        θ  = Vector{Float64}(undef, counts[k]),
        w  = Vector{Float64}(undef, counts[k]),
    ) for k in 1:nbands]
    cursors = zeros(Int, nbands)

    @inbounds for jp in 1:nZ, ip in 1:nR
        ilo = pix_ilo[ip, jp]
        ilo == 0 && continue
        r = pix_r[ip, jp]; θ = pix_θ[ip, jp]
        if ilo == -1
            k = 1; cursors[k] += 1
            tables[k].ip[cursors[k]] = ip; tables[k].jp[cursors[k]] = jp
            tables[k].r[cursors[k]]  = r;  tables[k].θ[cursors[k]]  = θ
            tables[k].w[cursors[k]]  = 1.0
        elseif ilo == -2
            k = nbands; cursors[k] += 1
            tables[k].ip[cursors[k]] = ip; tables[k].jp[cursors[k]] = jp
            tables[k].r[cursors[k]]  = r;  tables[k].θ[cursors[k]]  = θ
            tables[k].w[cursors[k]]  = 1.0
        else
            wlo = pix_wlo[ip, jp]
            whi = sqrt(max(1.0 - wlo*wlo, 0.0))
            k = ilo; cursors[k] += 1
            tables[k].ip[cursors[k]] = ip; tables[k].jp[cursors[k]] = jp
            tables[k].r[cursors[k]]  = r;  tables[k].θ[cursors[k]]  = θ
            tables[k].w[cursors[k]]  = wlo
            k = ilo + 1; cursors[k] += 1
            tables[k].ip[cursors[k]] = ip; tables[k].jp[cursors[k]] = jp
            tables[k].r[cursors[k]]  = r;  tables[k].θ[cursors[k]]  = θ
            tables[k].w[cursors[k]]  = whi
        end
    end
    return tables
end

band_pixel_tables = _build_band_pixel_tables(raster_geo_m, length(bands))
@info "per-band pixel tables built" nbands=length(bands) per_band_active=[length(t.ip) for t in band_pixel_tables]

# --- 5) Frame raster buffers (pre-allocated, NaN outside plasma) ---
# Active pixels (any band touches them) start at 0; everything else stays NaN
# so the heatmap shows the gap between plasma and frame.
active_mask_m = raster_geo_m.pix_ilo .!= 0
frame_rasters = [fill(NaN, nR_pix_multi, nZ_pix_multi) for _ in 1:n_frames_m]
for fr in frame_rasters
    @inbounds for jp in 1:nZ_pix_multi, ip in 1:nR_pix_multi
        active_mask_m[ip, jp] && (fr[ip, jp] = 0.0)
    end
end

# --- 6) Reusable kernels ---
# `_recon_band!` is kept as a fallback reference; the streaming pipeline below
# uses `_precompute_band_complex!` + `_recon_band_fast!` instead (Phase 1).
function _recon_band!(phi_rt::Matrix{Float64},
        x_rs::Vector{Float64}, θ_g::Vector{Float64},
        kx::Vector{Float64}, ky::Vector{Float64},
        ω_ky::Vector{Float64}, ψ0::Matrix{Float64},
        phi_amp::Matrix{Float64}, env::Matrix{Float64},
        pθ_const_θ::Vector{Float64}, kx_tilt_θ::Vector{Float64},
        ky_cap::Float64, t_norm::Float64)
    ntheta = length(θ_g); nky = length(ky); nkx = length(kx); nr = length(x_rs)
    fill!(phi_rt, 0.0)
    Threads.@threads for k in 1:ntheta
        @inbounds begin
            pθ_c   = pθ_const_θ[k]
            kx_tlt = kx_tilt_θ[k]
            for j in 1:nky
                envjk = env[k, j]
                envjk == 0 && continue
                kyj = ky[j]
                kyj > ky_cap && continue
                pθ  = kyj * pθ_c
                Δkx = kyj * kx_tlt
                δψ  = -ω_ky[j] * t_norm
                for ikx in 1:nkx
                    amp = phi_amp[ikx, j]
                    amp == 0 && continue
                    ktot = kx[ikx] + Δkx
                    ψ    = ψ0[ikx, j] + δψ
                    for ir in 1:nr
                        phi_rt[ir, k] += amp * cos(ktot * x_rs[ir] + pθ + ψ) * envjk
                    end
                end
            end
        end
    end
    return phi_rt
end

# Sum-of-angles split: cos(α − ω·t) = cos(α)·cos(ω·t) + sin(α)·sin(ω·t).
# Ar[j,ir,k] = env(k,j)·Σ_ikx amp·cos(α_ikx_j_ir_k); Ai analogous with sin.
# Layout: j is the FAST axis so the per-frame inner loop hits contiguous memory.
function _precompute_band_complex!(Ar::Array{Float64,3}, Ai::Array{Float64,3},
        x_rs::Vector{Float64}, θ_g::Vector{Float64},
        kx::Vector{Float64}, ky::Vector{Float64},
        ψ0::Matrix{Float64}, phi_amp::Matrix{Float64}, env::Matrix{Float64},
        pθ_const_θ::Vector{Float64}, kx_tilt_θ::Vector{Float64},
        ky_cap::Float64)
    ntheta = length(θ_g); nky = length(ky); nkx = length(kx); nr = length(x_rs)
    fill!(Ar, 0.0); fill!(Ai, 0.0)
    Threads.@threads for k in 1:ntheta
        @inbounds begin
            pθ_c   = pθ_const_θ[k]
            kx_tlt = kx_tilt_θ[k]
            for j in 1:nky
                envjk = env[k, j]
                envjk == 0 && continue
                kyj = ky[j]
                kyj > ky_cap && continue
                pθ  = kyj * pθ_c
                Δkx = kyj * kx_tlt
                for ikx in 1:nkx
                    amp = phi_amp[ikx, j]
                    amp == 0 && continue
                    ktot = kx[ikx] + Δkx
                    ψ    = ψ0[ikx, j]
                    a    = amp * envjk
                    for ir in 1:nr
                        ang = ktot * x_rs[ir] + pθ + ψ
                        Ar[j, ir, k] += a * cos(ang)
                        Ai[j, ir, k] += a * sin(ang)
                    end
                end
            end
        end
    end
    return nothing
end

# Per-frame: phi_rt[ir, k] = Σ_j (Ar[j,ir,k]·cos(ω[j]·t) + Ai[j,ir,k]·sin(ω[j]·t))
function _recon_band_fast!(phi_rt::Matrix{Float64},
        Ar::Array{Float64,3}, Ai::Array{Float64,3},
        ω_ky::Vector{Float64}, t_norm::Float64)
    nky, nr, ntheta = size(Ar)
    cω = Vector{Float64}(undef, nky); sω = Vector{Float64}(undef, nky)
    @inbounds for j in 1:nky
        cω[j] = cos(ω_ky[j] * t_norm)
        sω[j] = sin(ω_ky[j] * t_norm)
    end
    Threads.@threads for k in 1:ntheta
        @inbounds for ir in 1:nr
            s = 0.0
            for j in 1:nky
                s += Ar[j, ir, k] * cω[j] + Ai[j, ir, k] * sω[j]
            end
            phi_rt[ir, k] = s
        end
    end
    return phi_rt
end

# Per-band raster accumulation: evaluates the (r, θ) interpolator at this
# band's active pixels and adds `scale·weight·itp(r,θ)` into the frame raster.
# Threaded over the SoA pixel list. Each (ip,jp) appears at most ONCE in a
# single band's table, so no race within this call. Across bands we run
# serially (different iterations of the outer band loop), so the += is safe.
function _accumulate_band!(frame_raster::Matrix{Float64},
        ip_v::Vector{Int32}, jp_v::Vector{Int32},
        r_v::Vector{Float64}, θ_v::Vector{Float64}, w_v::Vector{Float64},
        itp, scale::Float64)
    n = length(ip_v)
    Threads.@threads for idx in 1:n
        @inbounds begin
            ip = ip_v[idx]; jp = jp_v[idx]
            frame_raster[ip, jp] += scale * w_v[idx] * itp(r_v[idx], θ_v[idx])
        end
    end
    return frame_raster
end

# --- 7) Band-streamed precompute → vmax pre-pass → production accumulation ---
# Bands processed SERIALLY so the heavy Ar/Ai allocation never overlaps;
# threading happens INSIDE `_precompute_band_complex!`, `_recon_band_fast!`,
# and `_accumulate_band!`. Peak heap = ONE band's Ar+Ai (~few GB) instead of
# n_bands × that. After each band we explicitly null + GC.gc() so the next
# band's allocation reuses the slot rather than swapping.
mkpath(out_dir_m)
vmax_per_band = fill(eps(Float64), length(bands))
t_setup_total = 0.0; t_pre_total = 0.0; t_recon_total = 0.0; t_accum_total = 0.0

for k in eachindex(bands)
    t0 = time()
    b      = bands[k]
    inp    = b.input; fs_k = b.fs; rs_a = b.rho_s_over_a
    rmin0  = Float64(inp.RMIN_LOC)
    q0     = Float64(abs(inp.Q_LOC))
    shat   = Float64(inp.Q_PRIME_LOC * rmin0^2 / q0^2)
    θw_ky  = collect(Float64, inp.WIDTH_SPECTRUM)
    fs_kx  = collect(Float64, fs_k.kx)
    fs_ky  = collect(Float64, fs_k.ky)

    r_center_lo = k == 1              ? rmin_edges[1]   : rmin_centers[k - 1]
    r_center_hi = k == length(bands)  ? rmin_edges[end] : rmin_centers[k + 1]
    r_lo  = max(r_center_lo - 0.01, 1e-3)
    r_hi  = min(r_center_hi + 0.01, 0.999)
    k_nyq_rs = π * rs_a / Δpix_a_glob
    ky_cap   = min(ky_cutoff_multi, k_nyq_rs)

    nkx_b = length(fs_kx); nky_b = length(fs_ky); ntheta_b = length(θ_g)
    r_g   = collect(range(r_lo, r_hi; length=nr_band))
    x_rs  = (r_g .- rmin0) ./ rs_a
    r0_rs = rmin0 / rs_a
    pθ_const_θ = -r0_rs .* θ_g
    kx_tilt_θ  = -shat  .* θ_g

    # Suffixed names (`env_b`, `phi_amp_b`) avoid clobbering globals of the
    # same name defined by upstream cells (cell 31's static torcut sets
    # `phi_amp` and `env_θky`; the bare `phi_amp` here would shadow it).
    env_b = Matrix{Float64}(undef, ntheta_b, nky_b)
    @inbounds for j in 1:nky_b, t in 1:ntheta_b
        env_b[t, j] = exp(-θ_g[t]^2 / (2 * θw_ky[j]^2))
    end
    phi_amp_b = sqrt.(max.(dropdims(sum(Array{Float64,3}(fs_k.phi2); dims=3); dims=3), 0))
    @inbounds for j in 1:nky_b, ikx in 1:nkx_b
        if abs(fs_kx[ikx]) > k_nyq_rs || fs_ky[j] > ky_cap
            phi_amp_b[ikx, j] = 0.0
        end
    end

    ω_ky_b = collect(Float64, band_ωky[k])
    phi_rt = zeros(Float64, nr_band, ntheta_b)

    Ar_b = zeros(Float64, nky_b, nr_band, ntheta_b)
    Ai_b = zeros(Float64, nky_b, nr_band, ntheta_b)
    GB = (length(Ar_b) + length(Ai_b)) * 8 / 1e9
    @info "band $k: starting precompute" ρ=b.ρ ntheta_b nky_b nr_band Ar_Ai_GB=round(GB, digits=2)

    t1 = time(); t_setup_total += t1 - t0
    _precompute_band_complex!(Ar_b, Ai_b, x_rs, θ_g, fs_kx, fs_ky,
        Matrix{Float64}(band_phases[k]), phi_amp_b, env_b,
        pθ_const_θ, kx_tilt_θ, Float64(ky_cap))
    t2 = time(); t_pre_total += t2 - t1

    # vmax pre-pass: track per-band peak |ϕ| over time
    vmax_k = 0.0
    for i in 1:n_frames_m
        _recon_band_fast!(phi_rt, Ar_b, Ai_b, ω_ky_b, t_seconds_m[i] * band_csa[k])
        vmax_k = max(vmax_k, maximum(abs, phi_rt))
    end
    vmax_per_band[k] = max(vmax_k, eps(Float64))

    # Per-band scaling. :global mode falls back to 1 here; we divide the
    # accumulated raster by vmax_global at the end (linear → factor pulls
    # out cleanly across all bands).
    band_scale = normalize_mode === :per_band ?
        (1.0 / vmax_per_band[k]) : 1.0

    # Production pass: per frame, recon → interp → accumulate band's contribution
    tbl = band_pixel_tables[k]
    t3 = time()
    for i in 1:n_frames_m
        _recon_band_fast!(phi_rt, Ar_b, Ai_b, ω_ky_b, t_seconds_m[i] * band_csa[k])
        # `Gridded(Linear())` stores a reference, so copy phi_rt — the next
        # iter overwrites it.
        itp_k = Interpolations.extrapolate(
            Interpolations.interpolate((r_g, θ_g), copy(phi_rt), Gridded(Linear())),
            (Flat(), Periodic()))
        _accumulate_band!(frame_rasters[i], tbl.ip, tbl.jp, tbl.r, tbl.θ, tbl.w,
            itp_k, band_scale)
    end
    t4 = time(); t_recon_total += t3 - t2; t_accum_total += t4 - t3

    # Free heavy precompute tables before moving on. Without explicit
    # `nothing` + `GC.gc()` the 1+ GB allocations can pile up before the
    # auto-collector kicks in, defeating the streaming plan.
    # Only Ar_b / Ai_b / phi_rt are GB-scale and worth nulling + GC.gc().
    # (The renamed env_b / phi_amp_b are KB-scale and would also have clobbered
    # upstream globals if we'd nulled them under their old bare names.)
    Ar_b = nothing; Ai_b = nothing; phi_rt = nothing
    GC.gc()
    @info "band $k done" ρ=b.ρ vmax=round(vmax_per_band[k], sigdigits=3) elapsed_s=round(t4 - t0, digits=2)
end

# --- 8) Optional global rescale (deferred for :global mode) ---
if normalize_mode === :global
    vmax_global = maximum(vmax_per_band)
    inv_vg = 1.0 / vmax_global
    Threads.@threads for i in 1:n_frames_m
        fr = frame_rasters[i]
        @inbounds for jp in 1:nZ_pix_multi, ip in 1:nR_pix_multi
            isnan(fr[ip, jp]) && continue
            fr[ip, jp] *= inv_vg
        end
    end
    @info "global rescale" vmax_global vmax_per_band
else
    @info "amplitude scan (per_band)" vmax_per_band
end
@info "band-streaming done" setup_s=round(t_setup_total, digits=2) precompute_s=round(t_pre_total, digits=2) recon_s=round(t_recon_total, digits=2) accumulate_s=round(t_accum_total, digits=2)

# --- 9) Plotting pass: pure rendering, no recon work ---
ρ_lo  = round(minimum(rho_list_multi), digits=2)
ρ_hi  = round(maximum(rho_list_multi), digits=2)
for (i, t_s) in enumerate(t_seconds_m)
    t_us  = t_s * 1e6
    t_str = @sprintf("%.1f", t_us)
    p_t = heatmap(Rpix_m, Zpix_m, frame_rasters[i]';
        c = cgrad(:RdBu_10, rev=true), clims = (-1.0, 1.0),
        aspect_ratio = :equal, colorbar = false, legend = false, framestyle = :box,
        xlabel = L"R / a", ylabel = L"Z / a", size = (540, 760),
        left_margin = 4mm, right_margin = 4mm, bottom_margin = 4mm, top_margin = 4mm,
        fontfamily = "Computer Modern",
        titlefontfamily = "Computer Modern",
        guidefontfamily = "Computer Modern",
        tickfontfamily  = "Computer Modern",
        title = L"\tilde{\varphi}\ \ (\rho \in [%$ρ_lo,\,%$ρ_hi])\quad t=%$t_str\,\mu s\quad [%$i/%$n_frames_m]")
    for kk in eachindex(bands)
        plot!(p_t, band_contour_R[kk], band_contour_Z[kk];
              color=:black, lw=0.4, ls=:dot)
    end
    plot!(p_t, outer_R, outer_Z; color=:black, lw=1.4)
    plot!(p_t, inner_R, inner_Z; color=:black, lw=1.4)
    savefig(p_t, joinpath(out_dir_m, @sprintf("frame_%04d.png", i)))
end
println("✔ wrote $n_frames_m PNGs to $out_dir_m")

# --- 10) FFmpeg → MP4 + GIF ---
ffmpeg_bin = let p = Sys.which("ffmpeg")
    if p !== nothing
        p
    else
        candidates = [
            joinpath(homedir(), "opt", "anaconda3", "envs", "ffmpeg", "bin", "ffmpeg"),
            joinpath(homedir(), "anaconda3",       "envs", "ffmpeg", "bin", "ffmpeg"),
            joinpath(homedir(), "miniconda3",      "envs", "ffmpeg", "bin", "ffmpeg"),
            joinpath(homedir(), "miniforge3",      "envs", "ffmpeg", "bin", "ffmpeg"),
        ]
        idx = findfirst(isfile, candidates)
        idx === nothing ? nothing : candidates[idx]
    end
end

if ffmpeg_bin !== nothing
    @info "Using ffmpeg" ffmpeg_bin
    mp4_m     = joinpath(out_dir_m, "movie.mp4")
    gif_m     = joinpath(out_dir_m, "movie.gif")
    pattern_m = joinpath(out_dir_m, "frame_%04d.png")
    try
        Base.run(`$ffmpeg_bin -y -loglevel error -framerate $mp4_fps_m -i $pattern_m -c:v libx264 -pix_fmt yuv420p -r $mp4_fps_m -crf 18 $mp4_m`)
        @info "MP4 written" mp4_m
        Base.run(`$ffmpeg_bin -y -loglevel error -framerate $mp4_fps_m -i $pattern_m -filter_complex "fps=$gif_fps_m,scale=$gif_width_m:-1:flags=lanczos,split[s0][s1];[s0]palettegen[p];[s1][p]paletteuse" -r $gif_fps_m $gif_m`)
        @info "GIF written" gif_m
        display("image/gif", read(gif_m))
    catch err
        @warn "ffmpeg failed" err
    end
else
    @warn "ffmpeg not found in PATH and no conda envs/ffmpeg/bin/ffmpeg — keeping the $n_frames_m PNGs in $out_dir_m"
end
